# HydroGEN data retrieval

In [10]:
# Import packages
from matplotlib import pyplot as plt
import pandas as pd
import hf_hydrodata as hf
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from pyproj import CRS
import os
from affine import Affine
import subprocess
from glob import glob

In [ ]:
# You need to register on https://hydrogen.princeton.edu/pin
# and run the following with your registered information
# before you can use the hydrodata utilities
hf.register_api_pin("XXX@uw.edu", "XXX")

#### Ma 2025

Water table depth

In [ ]:
# CONFIG
dataset  = "ma_2025"
variable = "water_table_depth"   
grid     = "conus2_wtd.30"

# big bbox
lat_min, lon_min = 46.3, -123.5
lat_max, lon_max = 47.8, -122.0

n_lat, n_lon = 8, 8
nodata = -9999.0

# native grid metadata (from docs)
RES = 24.14076631  # meters
X0  = -2848561.29  # meters (grid origin X, lower-left)
Y0  = -1724573.11  # meters (grid origin Y, lower-left)

CRS_LCC = CRS.from_proj4(
    "+proj=lcc +lat_1=30 +lat_2=60 "
    "+lon_0=-97.0 +lat_0=40.0000076294444 "
    "+a=6370000.0 +b=6370000"
)

out_dir = f"{variable}_tiles_lcc"
os.makedirs(out_dir, exist_ok=True)

# GRID-SPACE TILING
# 1) Convert the bbox to grid coords once
corners = [
    (lat_min, lon_min),
    (lat_min, lon_max),
    (lat_max, lon_min),
    (lat_max, lon_max),
]
gxs, gys = zip(*[hf.from_latlon(grid, la, lo) for la, lo in corners])

gx_min_all = int(np.floor(min(gxs)))
gx_max_all = int(np.ceil (max(gxs)))
gy_min_all = int(np.floor(min(gys)))
gy_max_all = int(np.ceil (max(gys)))

print("Whole bbox grid bounds:", gx_min_all, gy_min_all, gx_max_all, gy_max_all)

# 2) Build integer edges that are contiguous
gx_edges = np.linspace(gx_min_all, gx_max_all, n_lon + 1)
gy_edges = np.linspace(gy_min_all, gy_max_all, n_lat + 1)

# convert to int and force monotonic + exact endpoints
gx_edges = np.round(gx_edges).astype(int)
gy_edges = np.round(gy_edges).astype(int)

gx_edges[0], gx_edges[-1] = gx_min_all, gx_max_all
gy_edges[0], gy_edges[-1] = gy_min_all, gy_max_all

for k in range(len(gx_edges) - 1):
    if gx_edges[k+1] <= gx_edges[k]:
        gx_edges[k+1] = gx_edges[k] + 1
for k in range(len(gy_edges) - 1):
    if gy_edges[k+1] <= gy_edges[k]:
        gy_edges[k+1] = gy_edges[k] + 1

tile_paths = []

for i in range(n_lat):
    for j in range(n_lon):

        # Skip if tile exists
        tile_path = os.path.join(out_dir, f"{variable}_i{i:02d}_j{j:02d}.tif")
        if os.path.exists(tile_path):
            print("Tile exists, skipping:", tile_path)
            tile_paths.append(tile_path)
            continue

        gx_min, gx_max = int(gx_edges[j]), int(gx_edges[j+1])
        gy_min, gy_max = int(gy_edges[i]), int(gy_edges[i+1])

        span_x = gx_max - gx_min
        span_y = gy_max - gy_min

        options_gb = {
            "dataset": dataset,
            "variable": variable,
            "grid": grid,
            "grid_bounds": [gx_min, gy_min, gx_max, gy_max],  # [x_min, y_min, x_max, y_max]
        }

        arr = hf.get_gridded_data(options_gb).squeeze()
        arr = np.asarray(arr, dtype=np.float32)

        # rotation fix
        if arr.shape == (span_y, span_x):
            pass
        elif arr.shape == (span_x, span_y):
            arr = arr.T
        else:
            print(f"WARNING tile (i={i},j={j}) shape {arr.shape} vs expected {(span_y, span_x)}")

        # south-up -> north-up
        arr = np.flipud(arr)

        # nodata
        arr = np.where(np.isfinite(arr), arr, np.float32(nodata))

        # geotransform (meters): UL corner from gx_min, gy_max
        x_ul = X0 + gx_min * RES
        y_ul = Y0 + gy_max * RES
        transform = Affine(RES, 0, x_ul, 0, -RES, y_ul)

        tile_path = os.path.join(out_dir, f"{variable}_i{i:02d}_j{j:02d}.tif")
        with rasterio.open(
            tile_path, "w",
            driver="GTiff",
            height=arr.shape[0],
            width=arr.shape[1],
            count=1,
            dtype="float32",
            crs=CRS_LCC,
            transform=transform,
            nodata=np.float32(nodata),
            compress="deflate",
            tiled=True,
            blockxsize=256,
            blockysize=256,
            BIGTIFF="YES",
        ) as dst:
            dst.write(arr, 1)

        tile_paths.append(tile_path)
        print("Wrote:", tile_path)


print(f"\nTotal tiles: {len(tile_paths)} in {out_dir}")

# # MOSAIC WITH GDAL
# vrt_path = f"{variable}_mosaic.vrt"
# mosaic_lcc = f"{variable}_mosaic_lcc.tif"

# (A) build vrt
subprocess.run(
    ["gdalbuildvrt", "-overwrite", "-srcnodata", str(nodata), "-vrtnodata", str(nodata), vrt_path] + sorted(tile_paths),
    check=True
)

# (B) translate vrt -> tiled compressed GeoTIFF
subprocess.run(
    ["gdal_translate", vrt_path, mosaic_lcc,
     "-co", "TILED=YES", "-co", "COMPRESS=DEFLATE", "-co", "BIGTIFF=YES"],
    check=True
)

print("\nMosaic written:", mosaic_lcc)

# WARP TO EPSG:4326 (lat/lon)
mosaic_wgs84 = f"{variable}_mosaic_wgs84.tif"
subprocess.run(
    ["gdalwarp", "-overwrite",
     "-t_srs", "EPSG:4326",
     "-r", "bilinear",
     "-dstnodata", str(nodata),
     "-co", "TILED=YES", "-co", "COMPRESS=DEFLATE", "-co", "BIGTIFF=YES",
     mosaic_lcc, mosaic_wgs84],
    check=True
)

print("Warped (EPSG:4326) mosaic written:", mosaic_wgs84)


Uncertainty

In [ ]:
# CONFIG
dataset  = "ma_2025"
variable = "wtd_uncertainty"   
grid     = "conus2_wtd.30"

# big bbox
lat_min, lon_min = 46.3, -123.5
lat_max, lon_max = 47.8, -122.0

n_lat, n_lon = 8, 8
nodata = -9999.0

# native grid metadata (from docs)
RES = 24.14076631  # meters
X0  = -2848561.29  # meters (grid origin X, lower-left)
Y0  = -1724573.11  # meters (grid origin Y, lower-left)

CRS_LCC = CRS.from_proj4(
    "+proj=lcc +lat_1=30 +lat_2=60 "
    "+lon_0=-97.0 +lat_0=40.0000076294444 "
    "+a=6370000.0 +b=6370000"
)

out_dir = f"{variable}_tiles_lcc"
os.makedirs(out_dir, exist_ok=True)

# GRID-SPACE TILING
# 1) Convert the bbox to grid coords once
corners = [
    (lat_min, lon_min),
    (lat_min, lon_max),
    (lat_max, lon_min),
    (lat_max, lon_max),
]
gxs, gys = zip(*[hf.from_latlon(grid, la, lo) for la, lo in corners])

gx_min_all = int(np.floor(min(gxs)))
gx_max_all = int(np.ceil (max(gxs)))
gy_min_all = int(np.floor(min(gys)))
gy_max_all = int(np.ceil (max(gys)))

print("Whole bbox grid bounds:", gx_min_all, gy_min_all, gx_max_all, gy_max_all)

# 2) Build integer edges that are contiguous
gx_edges = np.linspace(gx_min_all, gx_max_all, n_lon + 1)
gy_edges = np.linspace(gy_min_all, gy_max_all, n_lat + 1)

# convert to int and force monotonic + exact endpoints
gx_edges = np.round(gx_edges).astype(int)
gy_edges = np.round(gy_edges).astype(int)

gx_edges[0], gx_edges[-1] = gx_min_all, gx_max_all
gy_edges[0], gy_edges[-1] = gy_min_all, gy_max_all

for k in range(len(gx_edges) - 1):
    if gx_edges[k+1] <= gx_edges[k]:
        gx_edges[k+1] = gx_edges[k] + 1
for k in range(len(gy_edges) - 1):
    if gy_edges[k+1] <= gy_edges[k]:
        gy_edges[k+1] = gy_edges[k] + 1

tile_paths = []

for i in range(n_lat):
    for j in range(n_lon):

        gx_min, gx_max = int(gx_edges[j]), int(gx_edges[j+1])
        gy_min, gy_max = int(gy_edges[i]), int(gy_edges[i+1])

        span_x = gx_max - gx_min
        span_y = gy_max - gy_min

        options_gb = {
            "dataset": dataset,
            "variable": variable,
            "grid": grid,
            "grid_bounds": [gx_min, gy_min, gx_max, gy_max],  # [x_min, y_min, x_max, y_max]
        }

        arr = hf.get_gridded_data(options_gb).squeeze()
        arr = np.asarray(arr, dtype=np.float32)

        # rotation fix
        if arr.shape == (span_y, span_x):
            pass
        elif arr.shape == (span_x, span_y):
            arr = arr.T
        else:
            print(f"WARNING tile (i={i},j={j}) shape {arr.shape} vs expected {(span_y, span_x)}")

        # south-up -> north-up
        arr = np.flipud(arr)

        # nodata
        arr = np.where(np.isfinite(arr), arr, np.float32(nodata))

        # geotransform (meters): UL corner from gx_min, gy_max
        x_ul = X0 + gx_min * RES
        y_ul = Y0 + gy_max * RES
        transform = Affine(RES, 0, x_ul, 0, -RES, y_ul)

        tile_path = os.path.join(out_dir, f"{variable}_i{i:02d}_j{j:02d}.tif")
        with rasterio.open(
            tile_path, "w",
            driver="GTiff",
            height=arr.shape[0],
            width=arr.shape[1],
            count=1,
            dtype="float32",
            crs=CRS_LCC,
            transform=transform,
            nodata=np.float32(nodata),
            compress="deflate",
            tiled=True,
            blockxsize=256,
            blockysize=256,
            BIGTIFF="YES",
        ) as dst:
            dst.write(arr, 1)

        tile_paths.append(tile_path)
        print("Wrote:", tile_path)


# print(f"\nTotal tiles: {len(tile_paths)} in {out_dir}")



In [ ]:
# MOSAIC WITH GDAL
vrt_path = f"{variable}_mosaic.vrt"
mosaic_lcc = f"{variable}_mosaic_lcc.tif"

# (A) build vrt
print(tile_paths)
subprocess.run(
    ["gdalbuildvrt", "-overwrite", "-srcnodata", str(nodata), "-vrtnodata", str(nodata), vrt_path] + sorted(tile_paths),
    check=True
)

# (B) translate vrt to tiled compressed GeoTIFF
subprocess.run(
    ["gdal_translate", vrt_path, mosaic_lcc,
     "-co", "TILED=YES", "-co", "COMPRESS=DEFLATE", "-co", "BIGTIFF=YES"],
    check=True
)

print("\nMosaic written:", mosaic_lcc)

# WARP TO EPSG:4326 (lat/lon)
mosaic_wgs84 = f"{variable}_mosaic_wgs84.tif"
subprocess.run(
    ["gdalwarp", "-overwrite",
     "-t_srs", "EPSG:4326",
     "-r", "bilinear",
     "-dstnodata", str(nodata),
     "-co", "TILED=YES", "-co", "COMPRESS=DEFLATE", "-co", "BIGTIFF=YES",
     mosaic_lcc, mosaic_wgs84],
    check=True
)

print("Warped (EPSG:4326) mosaic written:", mosaic_wgs84)

In [15]:
# import os

# # Remove variables that may point to the wrong GDAL/PROJ installation
# os.environ.pop("PROJ_LIB", None)
# os.environ.pop("PROJ_DATA", None)
# os.environ.pop("GDAL_DATA", None)

# import rasterio
# from rasterio.merge import merge
# from rasterio.warp import calculate_default_transform, reproject, Resampling

# mosaic_lcc = f"{variable}_mosaic_lcc.tif"
# mosaic_wgs84 = f"{variable}_mosaic_wgs84.tif"

# tile_paths_sorted = sorted(tile_paths)

# # Merge tiles
# srcs = [rasterio.open(fp) for fp in tile_paths_sorted]
# mosaic, out_transform = merge(srcs, nodata=nodata)

# meta = srcs[0].meta.copy()
# meta.update({
#     "driver": "GTiff",
#     "height": mosaic.shape[1],
#     "width": mosaic.shape[2],
#     "transform": out_transform,
#     "count": mosaic.shape[0],
#     "nodata": nodata,
#     "compress": "deflate",
#     "tiled": True,
# })

# with rasterio.open(mosaic_lcc, "w", **meta) as dst:
#     dst.write(mosaic)

# for src in srcs:
#     src.close()

# print("Mosaic written:", mosaic_lcc)

# # Reproject to WGS84
# dst_crs = "EPSG:4326"

# with rasterio.open(mosaic_lcc) as src:
#     print("Source CRS:", src.crs)

#     transform, width, height = calculate_default_transform(
#         src.crs, dst_crs, src.width, src.height, *src.bounds
#     )

#     kwargs = src.meta.copy()
#     kwargs.update({
#         "crs": dst_crs,
#         "transform": transform,
#         "width": width,
#         "height": height,
#         "nodata": nodata,
#         "compress": "deflate",
#         "tiled": True,
#     })

#     with rasterio.open(mosaic_wgs84, "w", **kwargs) as dst:
#         for i in range(1, src.count + 1):
#             reproject(
#                 source=rasterio.band(src, i),
#                 destination=rasterio.band(dst, i),
#                 src_transform=src.transform,
#                 src_crs=src.crs,
#                 dst_transform=transform,
#                 dst_crs=dst_crs,
#                 src_nodata=nodata,
#                 dst_nodata=nodata,
#                 resampling=Resampling.bilinear,
#             )

# print("Warped (EPSG:4326) mosaic written:", mosaic_wgs84)

In [16]:
# gwt_points_path = "Annual_Groundwater_Summary.csv"

# df = pd.read_csv(gwt_points_path)
# df

In [17]:
# # CONFIG
# dataset  = "ma_2025"
# variable1 = "water_table_depth"
# variable2 = "wtd_uncertainty"
# grid     = "conus2_wtd.30"
# nodata   = -9999.0

# # Locations

# results = []

# for _, row in df.iterrows():
#     site_id = row["Well_Tag_ID"]
#     lat = row["Latitude"]
#     lon = row["Longitude"]

#     # Convert lat/lon to grid coordinates
#     gx, gy = hf.from_latlon(grid, lat, lon)

#     # Use the containing integer grid cell
#     gx0 = int(np.floor(gx))
#     gy0 = int(np.floor(gy))

#     options = {
#         "dataset": dataset,
#         "variable": variable1,
#         "grid": grid,
#         "grid_bounds": [gx0, gy0, gx0 + 1, gy0 + 1],
#     }

#     try:
#         arr = hf.get_gridded_data(options).squeeze()
#         arr = np.asarray(arr, dtype=np.float32)

#         # Pull out scalar value
#         if arr.size == 1:
#             value1 = float(arr.ravel()[0])
#         else:
#             value1 = np.nan

#         if not np.isfinite(value1):
#             value1 = nodata

#     except Exception as e:
#         print(f"Failed for {site_id}: {e}")
#         value1 = nodata

#     try:
#         options["variable"] = variable2
#         arr2 = hf.get_gridded_data(options).squeeze()
#         arr2 = np.asarray(arr2, dtype=np.float32)

#         if arr2.size == 1:
#             value2 = float(arr2.ravel()[0])
#         else:
#             value2 = np.nan

#         if not np.isfinite(value2):
#             value2 = nodata

#     except Exception as e:
#         print(f"Failed for {site_id} variable2: {e}")
#         value2 = nodata

#     results.append({
#         "site_id": site_id,
#         "lat": lat,
#         "lon": lon,
#         variable1: value1,
#         variable2: value2
#     })

# out_df = pd.DataFrame(results)
# print(out_df)

# out_df.to_csv(f"Ma_point_samples.csv", index=False)